In [ ]:
### Import Libraries and Load Data
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

## Data Cleaning

In [ ]:
performance = pd.read_csv('NBA_Perf_22.csv',encoding='latin1')
performance["Player"] = performance["Player"].str.replace("?", "ć")
salary = pd.read_csv('nba_salaries_22.csv',encoding='utf-8-sig',engine='python')
salary['Salary'] = (
    salary['Salary']
      .str.replace(r'[\$,]', '', regex=True)
      .str.strip()
      .astype(float)
)

### Drop NAs, duplicates, and filter data
# Drop NAs
performance = performance.dropna()

In [ ]:
# 1. Identify which columns are “percent” stats vs. pure counts
num_cols = performance.select_dtypes(include="number").columns

In [ ]:
# 2. Group & sum everything, yielding one row per Player
performance = (
    performance
      .groupby("Player")[num_cols]
      .sum()
      .reset_index()
)

performance_norm = performance.copy()
# Drop the columns that are not needed
performance = performance.drop(columns=['Player','Age','FGA','3PA','2PA','FTA'])

# Exclude players who don't really play
def filter_played(df):
    df = df[df['MP'] > 10]
    df = df[df['G']>20]
    return df
performance = filter_played(performance)
### Standardize the variables
# Save the column and the index of the dataframe
col = performance.columns
index = performance.index

# now `agg` has exactly one row per Player  
# you can reset_index and carry on with scaling & k-Means:
#performance = agg.reset_index()

# Standardize the data
performance = performance.to_numpy()
performance_mean = np.mean(performance, axis=0)
performance_std = np.std(performance, axis=0)
performance=(performance-performance_mean)/performance_std
performance = pd.DataFrame(performance, columns= col, index=index)
### Run the clustering algo with your best guess for K
# Select Feature Data
featured_columns = ['PTS','eFG%','MP']
cluster_per = performance[featured_columns]
# Run the clustering algorithm with my best guess for K=3
kmeans_obj_performance = KMeans(n_clusters=3, random_state=1).fit(cluster_per)
### View the results
print(f'The cluster centers are {kmeans_obj_performance.cluster_centers_}')
print(f'The labels are {kmeans_obj_performance.labels_}')
print(f'The inertia is {kmeans_obj_performance.inertia_}')
### Create a visualization of the results with 2 or 3 variables that you think will best differentiate the clusters
fig = px.scatter_3d(cluster_per, x="MP", y="eFG%", z="PTS", color=kmeans_obj_performance.labels_, title="Points vs. eFG% vs. Minutes Played")
fig.show(renderer="vscode")

#I pick PTS as it is the most direct contributor to a player's contribution. 
#I chose eFG% because it takes the positional difference of players into account. Centers tend to have a higher percentage as they attack close to the rim, but not necessarily eFG%, as it is calculated using both 2PT and 3PT shots.
#Minutes Played is a good indicator of how much a player is used. A player who plays 30 minutes a game is likely to be more valuable than one who plays 10 minutes a game, even if they have the same PTS and eFG%.
### Evaluate the quality of the clustering using total variance explained and silhouette scores

#Total variance
X = cluster_per.values  # or clust_performance.to_numpy()

# 1. Total Sum of Squares (TSS)
tss = np.sum((X - np.mean(X, axis=0))**2)

# 2. Between‑cluster SS (BSS) = TSS – WSS (where WSS is inertia_)
wss = kmeans_obj_performance.inertia_
bss = tss - wss

# 3. Proportion of variance explained
var_explained = bss / tss

# 4. Silhouette score (single global value)
sil_score = silhouette_score(X, kmeans_obj_performance.labels_)

# 5. Print nicely
print(f'Total variance explained: {var_explained:.2%}')
print(f'Silhouette score: {sil_score:.3f}')
### Determine the ideal number of clusters using the elbow method and the silhouette coefficient
# elbow method
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=1)
    km.fit(X)
    wcss.append(km.inertia_)
elbow_df = pd.DataFrame({"k": range(1,11), "wcss": wcss})

# Compute silhouette scores for k = 2…10
sil_scores = []
ks = range(2, 11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=1)
    labels = km.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))
sil_df = pd.DataFrame({"k": list(ks), "silhouette": sil_scores})

# Determine best k by silhouette
best_sil_k = sil_df.loc[sil_df["silhouette"].idxmax(), "k"]
print(f"Best number of clusters by silhouette: {best_sil_k}")

# Plot elbow
fig1 = px.line(
    elbow_df, x="k", y="wcss", 
    title="Elbow Method"
)
fig1.update_layout(xaxis=dict(dtick=1))
fig1.show(renderer="vscode")

# Plot silhouette
fig2 = px.line(
    sil_df, x="k", y="silhouette", 
    title="Silhouette Scores"
)
fig2.update_layout(xaxis=dict(dtick=1))
fig2.show(renderer="vscode")

#On the elbow chart, we can see a sharp drop in WCSS (the within-cluster sum of squares) when we add 1 more cluster to the model from k=1. In general, adding clusters always causes over-fitting, so the goal is to have the least amount of clusters as possible while explaining the variance. In this case, the "elbow" of the chart is at k=2, meaning that when we have 2 clusters, the WCSS is relatively low and the model stays away from over-fitting.
### Use the recommended number of cluster (assuming it's different) to retrain your model and visualize the results
kmeans_obj_performance_best = KMeans(n_clusters=2, random_state=1).fit(cluster_per)
performance['cluster'] = kmeans_obj_performance_best.labels_
fig = px.scatter_3d(performance, x="MP", y="eFG%", z="PTS", color='cluster', title="eFG% vs. PTS vs. 3-point made votes for player performance")
fig.show(renderer="vscode")
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')

# axes data
x = performance['MP']
y = performance['eFG%']
z = performance['PTS']

# color by cluster (0/1)
c = performance['cluster'].astype(int)

scatter = ax.scatter(
    x, y, z,
    c=c,
    cmap='viridis',
    alpha=0.8
)

# labels & title
ax.set_xlabel('Minutes Played')
ax.set_ylabel('Effective FG%')
ax.set_zlabel('Points Scored')
ax.set_title('Minutes Played v. Effective FG% v. Points Scored')

# colorbar
cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label('Cluster Label')

plt.tight_layout()
plt.savefig('performance_clusters_3d.png', dpi=300)
plt.show()

### Once again evaluate the quality of the clustering using total variance explained and silhouette scores
#Once again evaluate the quality of the clustering using total variance explained and silhouette scores
#Total variance

X = cluster_per.values
# Total Sum of Squares (TSS)
tss = np.sum((X - np.mean(X, axis=0))**2)

#  Between‑cluster SS (BSS) = TSS – WSS (where WSS is inertia_)
wss = kmeans_obj_performance_best.inertia_
bss = tss - wss

#  Proportion of variance explained
var_explained = bss / tss

# Silhouette score (single global value)
sil_score = silhouette_score(X, kmeans_obj_performance_best.labels_)

print(f'Total variance explained: {var_explained:.2%}')
print(f'Silhouette score: {sil_score:.3f}')

### Use the model to select players for Mr. Rooney to consider
#The objective of this project is to locate players who perform well but are not as costly as the super stars. If we use 2 clusters, we will only have 2 groups of players: the super stars who exceed in all three metrics and the others who have mediocre performance. We want a middle group of players who are not super stars but are still good enough to be considered. Therefore, we will use 3 clusters instead of 2.**
import unicodedata
performance['Player'] = performance_norm.loc[index, 'Player'].values
def normalize_name(name):
    name = str(name)
    nfkd = unicodedata.normalize("NFKD", name)
    stripped = "".join(c for c in nfkd if not unicodedata.combining(c))
    return stripped.lower().strip()

# Create normalized‐name columns
performance['Player_norm'] = performance['Player'].map(normalize_name)
salary['Player_norm'] = salary['Player'].map(normalize_name)

#. Merge salary into performance
performance = performance.merge(
    salary[['Player_norm','Salary']],
    on='Player_norm',
    how='left'
)

# 4. Fill any missing salaries
#median_sal = salary['Salary'].median()
#performance['Salary'] = performance['Salary'].fillna(median_sal)
performance= performance.dropna()
cluster_per_selected = performance[['PTS','eFG%','MP']]
kmeans_obj_performance_selected = KMeans(n_clusters=3, random_state=1).fit(cluster_per_selected)
performance['cluster_real'] = kmeans_obj_performance_selected.labels_

fig = px.scatter_3d(
    performance,
    x="MP", y="eFG%", z="PTS",
    color='cluster_real', 
    size = "Salary",
    size_max=20,
    hover_data=['Player','Salary'],
    title="Minutes Played vs. Efficient FG% vs. Points Scored (Size by Salary)")
fig.show(renderer="vscode")
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')

# axes data
x = performance['MP']
y = performance['eFG%']
z = performance['PTS']

# color by cluster (0/1)
c = performance['cluster_real'].astype(int)

scatter = ax.scatter(
    x, y, z,
    c=c,
    cmap='viridis',
    alpha=0.8
)

# labels & title
ax.set_xlabel('Minutes Played')
ax.set_ylabel('Points per Game')
ax.set_zlabel('Effective FG%')
ax.set_title('Minutes Played vs. Efficient FG% vs. Points Scored (Size by Salary)')

# colorbar
cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label('Cluster Label')

plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')

# axes data
x = performance['MP']
y = performance['eFG%']
z = performance['PTS']

# color by cluster (0/1)
c = performance['cluster_real'].astype(int)

# scale marker size by salary
sizes = (performance['Salary'] / performance['Salary'].max()) * 100  # tweak max marker size

scatter = ax.scatter(
    x, y, z,
    c=c,
    cmap='viridis',
    s=sizes,
    alpha=0.8
)

# labels & title
ax.set_xlabel('Minutes Played')
ax.set_ylabel('Points per Game')
ax.set_zlabel('Effective FG%')
ax.set_title('Minutes Played vs. Efficient FG% vs. Points Scored (Size by Salary)')

# colorbar
cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label('Cluster Label')

plt.tight_layout()
plt.show()

from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd


# Cluster on the 3 performance stats
kmeans = KMeans(n_clusters=3, random_state=1, algorithm="lloyd")
performance["cluster"] = kmeans.fit_predict(
    performance[["PTS","eFG%","3P"]]
)

# Prepare the tree DataFrame
tree_data = performance[["Salary","cluster"]].copy()
# If you want to include the raw stats as well:
# tree_data = performance[["Salary","PTS","eFG%","3P","cluster"]].copy()

# Convert cluster to categorical (and dummies)
tree_data["cluster"] = tree_data["cluster"].astype("category")
tree_data = pd.get_dummies(tree_data, columns=["cluster"], drop_first=True)

#  Train/test split
X = tree_data.drop(columns=["Salary"])
y = tree_data["Salary"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

#  Fit the Decision Tree Regressor
dt_reg = DecisionTreeRegressor(max_depth=4, random_state=1)
dt_reg.fit(X_train, y_train)

# Predict & evaluate
y_pred = dt_reg.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.0f}")
print(f"Test R²:   {r2_score(y_test, y_pred):.3f}")

# Compute residuals on the full dataset (re-fit to all data)
dt_full = DecisionTreeRegressor(max_depth=4, random_state=1)
dt_full.fit(X, y)
performance["salary_pred"] = dt_full.predict(X)
performance["residual"]      = performance["Salary"] - performance["salary_pred"]

# Define a dynamic 5% threshold per player
threshold = 0.05 * performance["salary_pred"]

# Build conditions
conds = [
    performance["residual"] <= -threshold,   # actual << predicted → underpaid
    performance["residual"] >=  threshold    # actual >> predicted → overpaid
]
choices = ["underpaid", "overpaid"]

# Assign “fair” by default
performance["pay_category_dt"] = np.select(conds, choices, default="fair")

# Inspect all three groups
for cat in ["underpaid","fair","overpaid"]:
    print(f"\n=== {cat.upper()} ({cat=='fair' and 'close to predicted' or ''}) ===")
    display(
      performance
        .loc[performance["pay_category_dt"]==cat, 
             ["Player","Salary","salary_pred","residual"]]
        .sort_values(by="residual", ascending=(cat!="overpaid"))
        .head(4)
    )

### Another way (performance-index)
from scipy.stats import zscore

# compute z‑scores column‑wise
zs = performance[featured_columns].apply(zscore)
zs.columns = [f"z_{c}" for c in featured_columns]
performance = performance.join(zs)
performance["perf_index"] = performance[[f"z_{c}" for c in featured_columns]].sum(axis=1)

# Compute cluster medians for perf_index and Salary
cluster_stats = performance.groupby("cluster").agg(
    med_perf   = ("perf_index", "median"),
    med_salary = ("Salary",      "median")
)

# Join those stats back onto each player
performance = performance.join(cluster_stats, on="cluster")

# Classify pay category:
conds = [
    # under‑paid: you perform ≥ your cluster’s median but cost ≤ its median
    (performance["perf_index"] >= performance["med_perf"]) &
    (performance["Salary"]     <= performance["med_salary"]),

    # over‑paid: you perform ≤ your cluster’s median but cost ≥ its median
    (performance["perf_index"] <= performance["med_perf"]) &
    (performance["Salary"]     >= performance["med_salary"])
]
choices = ["underpaid", "overpaid"]

performance["pay_category"] = np.select(conds, choices, default="fair")

# Quick look at your three groups
for cat in ["underpaid","fair","overpaid"]:
    print(f"\n {cat.upper()} PLAYERS: ")
    display(
        performance
          .loc[performance["pay_category"]==cat, 
               ["Player","cluster","perf_index","Salary"]]
          .sort_values(["perf_index","Salary"], ascending=[False, True])
          .head(3)
    )
### *Another way to measure a player's performance: PER (Player Efficiency Rating)
# Reload the dataset
performance = pd.read_csv('../data/NBA_Perf_22.csv',encoding='latin1')
performance["Player"] = performance["Player"].str.replace("?", "ć")
# Calculate the PER with the PER formula
performance['PER']= (performance['FG']*85.910
                     +performance['STL']*53.897
                     +performance['3P']*51.757
                     +performance['FT']*46.845
                     +performance['BLK']*39.190
                     +performance['ORB']*39.190
                     +performance['DRB']*14.707
                     +performance['AST']*34.677
                     -(performance['FGA']-performance['FG'])*39.190
                     -(performance['FTA']-performance['FT'])*20.091
                     -performance['TOV']*53.897
                     -performance['PF']*17.174)*(1/performance['MP'])
performance['PER'] = performance['PER'].round(2)
performance['PER'] = performance['PER'].replace([np.inf, -np.inf], np.nan)
performance['PER'] = performance['PER'].fillna(0)
performance['PER'] = performance['PER'].astype(float)
def filter_played(df):
    df = df[df['MP'] > 10]
    df = df[df['G']>20]
    return df
performance = filter_played(performance)
top_10_players = performance.sort_values(by='PER', ascending=False).head(10)
print(top_10_players[['Player', 'PER']])